# ABSA Fine-Tuning Token Budget

This notebook measures the exact Qwen tokenizer budget needed to fine-tune with `prompt + gold JSON` examples.

Main questions:

- How many tokens does `absa_v6` use before the answer starts?
- How many tokens are needed for `prompt + gold JSON` on train/devel?
- Which `max_length` keeps the answer visible without truncation?
- Where should labels switch from `-100` to trainable assistant tokens?

Run this notebook on the machine where `ABSA/model` contains `Qwen/Qwen3.5-2B`, or let `transformers` download only the tokenizer from Hugging Face.

In [ ]:
from pathlib import Path
import json
import math
from statistics import mean

import numpy as np
import pandas as pd

try:
    from transformers import AutoTokenizer
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Install transformers first, for example: pip install transformers accelerate sentencepiece"
    ) from exc


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / "distribucio-codi-i-dades-12-03-26" / "ABSA").exists():
            return path
    raise FileNotFoundError("Could not find repository root from current working directory")


ROOT = find_repo_root()
ABSA_DIR = ROOT / "distribucio-codi-i-dades-12-03-26" / "ABSA"
MODEL_DIR = ABSA_DIR / "model"
PROMPT_PATH = ABSA_DIR / "prompts" / "absa_v6.json"
TRAIN_PATH = ABSA_DIR / "dataset" / "train.json"
DEVEL_PATH = ABSA_DIR / "dataset" / "devel.json"

print("ROOT:", ROOT)
print("ABSA_DIR:", ABSA_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("PROMPT_PATH:", PROMPT_PATH)

## Load tokenizer

The exact counts must come from the same tokenizer used by the fine-tuning and inference scripts. The notebook prefers the local `ABSA/model` folder. If that folder does not contain tokenizer files, it falls back to the public model id.

In [ ]:
MODEL_ID = "Qwen/Qwen3.5-2B"
local_tokenizer_exists = any(
    (MODEL_DIR / name).exists()
    for name in ["tokenizer.json", "tokenizer.model", "tokenizer_config.json"]
)
TOKENIZER_SOURCE = str(MODEL_DIR if local_tokenizer_exists else MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_SOURCE, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("Tokenizer source:", TOKENIZER_SOURCE)
print("Tokenizer class:", tokenizer.__class__.__name__)
print("eos_token:", repr(tokenizer.eos_token), tokenizer.eos_token_id)
print("pad_token:", repr(tokenizer.pad_token), tokenizer.pad_token_id)
print("chat_template exists:", tokenizer.chat_template is not None)

In [ ]:
ASPECTS = [
    "restaurant_general",
    "restaurant_prices",
    "food_quality",
    "food_prices",
    "food_style_options",
    "drinks_quality",
    "drinks_prices",
    "drinks_style_options",
    "ambience",
    "service",
    "location",
]
POLARITIES = ["positive", "negative", "neutral", "conflict"]


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def render_template(template, values):
    text = template
    for key, value in values.items():
        text = text.replace("{" + key + "}", str(value))
    return text


def prompt_values(example):
    return {
        "text": example["text"],
        "language": example.get("language", "unknown"),
        "aspects": ", ".join(ASPECTS),
        "polarities": ", ".join(POLARITIES),
    }


def prepare_prompt_messages(prompts, example):
    values = prompt_values(example)
    return [
        {"role": "system", "content": render_template(prompts["system"], values)},
        {"role": "user", "content": render_template(prompts["user"], values)},
    ]


def gold_json(example, compact=True):
    if compact:
        return json.dumps(
            example.get("gold", {}),
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
        )
    return json.dumps(example.get("gold", {}), ensure_ascii=False)


def apply_chat_template(messages, add_generation_prompt=False):
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )


def encode_text(text):
    return tokenizer(text, add_special_tokens=False)["input_ids"]


def longest_common_prefix(xs, ys):
    limit = min(len(xs), len(ys))
    for i in range(limit):
        if xs[i] != ys[i]:
            return i
    return limit


prompts_raw = load_json(PROMPT_PATH)
prompts = {
    "name": prompts_raw.get("name", PROMPT_PATH.stem),
    "system": prompts_raw.get("system") or prompts_raw.get("sysprompt"),
    "user": prompts_raw.get("user") or prompts_raw.get("usrprompt"),
}
train_examples = load_json(TRAIN_PATH)
devel_examples = load_json(DEVEL_PATH)

print("Prompt:", prompts["name"])
print("System chars:", len(prompts["system"]))
print("User template chars:", len(prompts["user"]))
print("Train examples:", len(train_examples))
print("Devel examples:", len(devel_examples))

## Static prompt budget

This estimates how much of the context is consumed by the fixed prompt itself before adding a real review. The exact training budget is computed per example below.

In [ ]:
blank_example = {"text": "", "language": "es", "gold": {}}
blank_messages = prepare_prompt_messages(prompts, blank_example)
blank_prompt_text = apply_chat_template(blank_messages, add_generation_prompt=True)
blank_prompt_ids = encode_text(blank_prompt_text)

system_ids = encode_text(prompts["system"])
user_template_ids = encode_text(render_template(prompts["user"], prompt_values(blank_example)))

print("system content tokens:", len(system_ids))
print("user template content tokens with empty review:", len(user_template_ids))
print("chat-formatted prompt tokens with empty review:", len(blank_prompt_ids))
print("chat-formatted prompt tail:")
print(blank_prompt_text[-1200:])

## Per-example token accounting

`prompt_tokens` is the masked prefix: system + user + assistant generation header. `target_tokens_*` is the gold JSON answer. For correct SFT, labels should be `-100` for the first `prompt_tokens` tokens and normal token ids only for the target JSON tokens.

In [ ]:
def token_account(example, dataset_name, compact=True):
    prompt_messages = prepare_prompt_messages(prompts, example)
    answer = gold_json(example, compact=compact)
    full_messages = prompt_messages + [{"role": "assistant", "content": answer}]

    prompt_text = apply_chat_template(prompt_messages, add_generation_prompt=True)
    full_text = apply_chat_template(full_messages, add_generation_prompt=False)

    prompt_ids = encode_text(prompt_text)
    full_ids = encode_text(full_text)
    lcp = longest_common_prefix(prompt_ids, full_ids)
    prefix_matches = lcp == len(prompt_ids)

    target_from_prefix = full_ids[len(prompt_ids):] if prefix_matches else full_ids[lcp:]
    direct_answer_ids = encode_text(answer)

    return {
        "dataset": dataset_name,
        "id": example.get("id"),
        "language": example.get("language", "unknown"),
        "text_chars": len(example.get("text", "")),
        "gold_labels": len(example.get("gold", {})),
        "gold_json_chars": len(answer),
        "prompt_tokens": len(prompt_ids),
        "full_tokens": len(full_ids),
        "target_tokens_from_template": len(target_from_prefix),
        "target_tokens_direct_json": len(direct_answer_ids),
        "prefix_lcp": lcp,
        "prefix_matches": prefix_matches,
        "prompt_text": prompt_text,
        "full_text": full_text,
        "prompt_ids": prompt_ids,
        "full_ids": full_ids,
        "answer": answer,
    }


records_compact = [
    token_account(ex, "train", compact=True) for ex in train_examples
] + [
    token_account(ex, "devel", compact=True) for ex in devel_examples
]
records_current = [
    token_account(ex, "train", compact=False) for ex in train_examples
] + [
    token_account(ex, "devel", compact=False) for ex in devel_examples
]

df = pd.DataFrame([{k: v for k, v in r.items() if not k.endswith("text") and not k.endswith("ids")} for r in records_compact])
df_current = pd.DataFrame([{k: v for k, v in r.items() if not k.endswith("text") and not k.endswith("ids")} for r in records_current])

print("compact prefix matches:", df["prefix_matches"].mean(), "bad:", int((~df["prefix_matches"]).sum()))
print("current-json prefix matches:", df_current["prefix_matches"].mean(), "bad:", int((~df_current["prefix_matches"]).sum()))
df.head()

In [ ]:
PERCENTILES = [0, 1, 5, 10, 25, 50, 75, 90, 95, 97.5, 99, 100]
TOKEN_COLUMNS = [
    "prompt_tokens",
    "full_tokens",
    "target_tokens_from_template",
    "target_tokens_direct_json",
]


def percentile_table(frame, columns=TOKEN_COLUMNS):
    rows = []
    for dataset_name, group in frame.groupby("dataset", sort=False):
        for col in columns:
            values = group[col].to_numpy()
            row = {"dataset": dataset_name, "metric": col, "n": len(values), "mean": values.mean()}
            for p in PERCENTILES:
                row[f"p{p}"] = np.percentile(values, p)
            rows.append(row)
    return pd.DataFrame(rows)


summary_compact = percentile_table(df)
summary_current = percentile_table(df_current)

print("Compact gold JSON percentiles")
display(summary_compact.round(2))

print("Current json.dumps default JSON percentiles")
display(summary_current.round(2))

In [ ]:
MAX_LENGTHS = [512, 768, 1024, 1280, 1536, 1792, 2048, 2560, 3072, 4096]


def fit_table(frame):
    rows = []
    for dataset_name, group in frame.groupby("dataset", sort=False):
        for max_len in MAX_LENGTHS:
            rows.append({
                "dataset": dataset_name,
                "max_length": max_len,
                "prompt_fit_%": 100 * (group["prompt_tokens"] <= max_len).mean(),
                "full_fit_%": 100 * (group["full_tokens"] <= max_len).mean(),
                "answer_visible_%": 100 * (group["prompt_tokens"] < max_len).mean(),
                "examples_truncated": int((group["full_tokens"] > max_len).sum()),
            })
    return pd.DataFrame(rows)


display(fit_table(df).round(2))

needed_p99 = math.ceil(np.percentile(df[df["dataset"] == "train"]["full_tokens"], 99))
needed_p100 = int(df[df["dataset"] == "train"]["full_tokens"].max())
print("Train compact full_tokens p99:", needed_p99)
print("Train compact full_tokens max:", needed_p100)
print("Smallest tested max_length with 100% train fit:", next((x for x in MAX_LENGTHS if x >= needed_p100), None))

## Inspect the masking boundary

Use this section to verify exactly which tokens are masked and which tokens are trained. The expected result is that all system/user/chat-header tokens are masked, and only the final assistant JSON remains trainable.

In [ ]:
def token_rows(token_ids, start=0, limit=120):
    rows = []
    for offset, token_id in enumerate(token_ids[start:start + limit], start=start):
        piece = tokenizer.decode([token_id], skip_special_tokens=False)
        rows.append({"pos": offset, "id": token_id, "piece": repr(piece)})
    return pd.DataFrame(rows)


def inspect_example(record, around=40):
    prompt_len = len(record["prompt_ids"])
    full_ids = record["full_ids"]
    lcp = longest_common_prefix(record["prompt_ids"], full_ids)
    answer_start = prompt_len if lcp == prompt_len else lcp

    labels = [-100] * answer_start + full_ids[answer_start:]
    print("id:", record["id"])
    print("language:", record["language"])
    print("gold JSON:", record["answer"])
    print("prompt_len:", prompt_len)
    print("full_len:", len(full_ids))
    print("answer_start:", answer_start)
    print("prefix_matches:", lcp == prompt_len)
    print("trainable_tokens:", sum(x != -100 for x in labels))
    print("decoded trainable span:")
    print(tokenizer.decode(full_ids[answer_start:], skip_special_tokens=False))

    start = max(0, answer_start - around)
    end_limit = around * 2
    display(token_rows(full_ids, start=start, limit=end_limit))


longest = max(records_compact, key=lambda r: r["full_tokens"])
median_id = df[df["dataset"] == "train"].sort_values("full_tokens").iloc[len(train_examples) // 2]["id"]
median_record = next(r for r in records_compact if r["id"] == median_id)

print("Median-length training example")
inspect_example(median_record)

print("Longest training/devel example")
inspect_example(longest)

## Candidate label construction for training

This cell mirrors what the training collator should do later. It does not train anything; it only builds `input_ids` and `labels` for one example and checks that the label mask is correct.

In [ ]:
def build_sft_features(example, max_length=None, compact=True):
    record = token_account(example, "single", compact=compact)
    prompt_len = len(record["prompt_ids"])
    full_ids = record["full_ids"]
    lcp = longest_common_prefix(record["prompt_ids"], full_ids)
    if lcp != prompt_len:
        print("Warning: prompt ids are not an exact prefix; using LCP as answer_start", lcp)
        answer_start = lcp
    else:
        answer_start = prompt_len

    input_ids = full_ids[:]
    labels = [-100] * answer_start + full_ids[answer_start:]

    if max_length is not None and len(input_ids) > max_length:
        # Left truncation preserves the final assistant JSON. If max_length is
        # chosen from the p100 table above this branch should not be needed.
        overflow = len(input_ids) - max_length
        input_ids = input_ids[overflow:]
        labels = labels[overflow:]
        answer_start = max(0, answer_start - overflow)

    return {
        "input_ids": input_ids,
        "labels": labels,
        "answer_start": answer_start,
        "trainable_tokens": sum(x != -100 for x in labels),
        "truncated": max_length is not None and len(full_ids) > max_length,
    }


features = build_sft_features(train_examples[0], max_length=None, compact=True)
print("input tokens:", len(features["input_ids"]))
print("answer_start:", features["answer_start"])
print("trainable_tokens:", features["trainable_tokens"])
print("truncated:", features["truncated"])
print("decoded trained target:")
print(tokenizer.decode([x for x in features["labels"] if x != -100], skip_special_tokens=False))